# NLP Project: SMS Spam Detection using Average Word2Vec
## Industry-Style Notebook
This notebook explains every step from problem statement to deployment-style prediction.

## 1. Problem Statement

Explain spam detection, business value, challenges.


## 2. Learning Objectives

NLP pipeline, embeddings, Random Forest.


## 3. Dataset Overview

Describe SMSSpamCollection and labels.


## 4. Exploratory Data Analysis

Inspect shape, class balance, examples.


## 5. Text Preprocessing Theory

Cleaning, lowercasing, lemmatization.


## 6. Tokenization

Convert text to tokens.


## 7. Word2Vec Theory

CBOW, Skip-gram, vector_size, window, min_count, epochs.


## 8. Train Word2Vec

Train embeddings from corpus.


## 9. Average Word2Vec

Average word vectors into sentence vector.


## 10. Feature Matrix

Create X and validate shapes.


## 11. Target Variable

Create y.


## 12. Train/Test Split

Why split, stratify.


## 13. Random Forest Theory

Bootstrap, trees, voting.


## 14. Model Training

Fit classifier.


## 15. Evaluation

Accuracy, Precision, Recall, F1, Confusion Matrix.


## 16. Predict New Messages

Inference pipeline.


## 17. Common Errors

NaN vectors, shape mismatch, append removal, leakage.


## 18. Interview Questions

Conceptual questions.


## 19. Exercises

Practice tasks.


In [1]:
# Core implementation
import re, numpy as np, pandas as pd, nltk, gensim
from tqdm import tqdm
from nltk.stem import WordNetLemmatizer
from gensim.utils import simple_preprocess
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
nltk.download('wordnet')
messages=pd.read_csv('SMSSpamCollection',sep='\t',names=['label','message'])
lemmatizer=WordNetLemmatizer()
corpus=[]
for text in messages.message:
    txt=re.sub('[^a-zA-Z]',' ',text).lower().split()
    txt=[lemmatizer.lemmatize(w) for w in txt]
    corpus.append(' '.join(txt))
words=[simple_preprocess(t) for t in corpus]
model=gensim.models.Word2Vec(sentences=words,vector_size=100,window=5,min_count=1,epochs=10,seed=42)
def avg_word2vec(doc):
    vecs=[model.wv[w] for w in doc if w in model.wv]
    return np.mean(vecs,axis=0) if vecs else np.zeros(model.vector_size)
X=np.array([avg_word2vec(d) for d in tqdm(words)])
y=pd.get_dummies(messages.label).iloc[:,0].values
print(messages.shape,X.shape,y.shape,np.isnan(X).sum())
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
clf=RandomForestClassifier(random_state=42)
clf.fit(X_train,y_train)
pred=clf.predict(X_test)
print(accuracy_score(y_test,pred))
print(classification_report(y_test,pred))
print(confusion_matrix(y_test,pred))
def predict_sms(msg):
    txt=re.sub('[^a-zA-Z]',' ',msg).lower().split()
    txt=[lemmatizer.lemmatize(w) for w in txt]
    tokens=simple_preprocess(' '.join(txt))
    p=clf.predict(avg_word2vec(tokens).reshape(1,-1))[0]
    return 'Spam' if p==1 else 'Ham'
for m in ['Congratulations! You won $1000','See you at 6 pm','URGENT claim your prize']:
    print(m,'->',predict_sms(m))


[nltk_data] Downloading package wordnet to
[nltk_data]     c:\DS2026\Python\.venv\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
100%|██████████| 5572/5572 [00:01<00:00, 4194.49it/s]


(5572, 2) (5572, 100) (5572,) 0
0.9668161434977578
              precision    recall  f1-score   support

       False       0.91      0.83      0.87       149
        True       0.97      0.99      0.98       966

    accuracy                           0.97      1115
   macro avg       0.94      0.91      0.93      1115
weighted avg       0.97      0.97      0.97      1115

[[124  25]
 [ 12 954]]
Congratulations! You won $1000 -> Spam
See you at 6 pm -> Spam
URGENT claim your prize -> Ham
